In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors flask
!pip install -q huggingface_hub
!npm install -g localtunnel > /dev/null 2>&1

In [ ]:
# Cell 2: Download model
import torch
from diffusers import StableDiffusionPipeline

print('Downloading model...')
model_id = 'stablediffusionapi/anything-v5'
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()
print('Model loaded on GPU!')

In [ ]:
# Cell 3: Start API server + tunnel
from flask import Flask, request, send_file, jsonify
from flask import Response
import io, subprocess, threading, time, re

app = Flask(__name__)

@app.after_request
def add_cors(response):
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = '*'
    response.headers['Access-Control-Allow-Methods'] = '*'
    return response

@app.route('/generate', methods=['POST', 'OPTIONS'])
def generate():
    if request.method == 'OPTIONS':
        return Response(status=200)
    data = request.json
    prompt = data.get('prompt', '')
    negative = data.get('negative_prompt', 'ugly, deformed, blurry, low quality, bad anatomy, censored, clothed, text, watermark')
    steps = min(data.get('steps', 25), 50)
    width = data.get('width', 512)
    height = data.get('height', 512)
    full_prompt = f'{prompt}, masterpiece, best quality, detailed, anime, hentai'
    image = pipe(
        full_prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=7.5,
        width=width,
        height=height
    ).images[0]
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    buf.seek(0)
    return send_file(buf, mimetype='image/png')

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': 'anything-v5'})

# Start Flask in background
def run_flask():
    app.run(port=5000, host='0.0.0.0')

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

# Start localtunnel
proc = subprocess.Popen(['lt', '--port', '5000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
time.sleep(5)
line = proc.stdout.readline()
print(f'\n🔥 API is live!')
print(f'URL: {line.strip()}')
print('\nCopy this URL into the web app!')

# Keep alive
while True:
    time.sleep(60)